**Persiapan: SparkSession**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window
import pandas as pd

spark = SparkSession.builder.appName("Tugas5-DashboardCabang").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/19 23:35:45 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/19 23:35:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 23:35:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**1. Baca data transaksi**

Baca transaksi_tugas5.csv, lalu buat kolom pendapatan = unit_terjual x harga_satuan.

In [2]:
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

df_transaksi.printSchema()
df_transaksi.show(5)

root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



**2. Buat df_target**

Tabel referensi target bulanan & PIC per cabang kota.

In [3]:
# Tabel target & PIC per cabang kota
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



**A. Join & Perbandingan Target**

Ringkas total pendapatan per kota dari df_transaksi, lalu join dengan df_target. Tambahkan kolom pencapaian_persen. Urutkan hasil dari pencapaian tertinggi.

Pendekatan: agregasi dulu (5 baris kota) baru join ke df_target — jauh lebih efisien daripada join dulu baru agregasi, seperti dicontohkan di modul (Sub-bab 5.1.2).

In [4]:
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100) \
    .orderBy(col("pencapaian_persen").desc())

hasil_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota**

Menggunakan window function, tentukan kategori dengan pendapatan tertinggi di setiap kota (top-1 saja, gunakan row_number()).

Pendekatan: ringkas total pendapatan per kombinasi kota + kategori terlebih dahulu, baru terapkan window partitionBy("kota") untuk mengambil baris teratas per kota.

In [5]:
ringkasan_kat_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

window_b = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

hasil_b = ringkasan_kat_kota.withColumn("rn", row_number().over(window_b)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy(col("total_pendapatan").desc())

hasil_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|Yogyakarta|             Fashion|        13325000|
|  Semarang|        Rumah Tangga|        11125000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|      Solo|Kesehatan & Kecan...|         8425000|
|  Magelang|Kesehatan & Kecan...|         7275000|
+----------+--------------------+----------------+



**C. Spark SQL**

Daftarkan df_transaksi dan df_target sebagai temporary view, lalu tulis satu kueri SQL (bukan DataFrame API) yang menampilkan: kota, pic_cabang, dan jumlah transaksi (COUNT) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [6]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

hasil_c = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_c.show()

[Stage 16:>                                                         (0 + 4) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan**

Kalau lihat angkanya, cabang terbaik jelas Purworejo: pendapatannya Rp45,65 juta, alias 152% dari target Rp30 juta, dan cuma dia yang berhasil lewatin target. Tapi ya, targetnya emang paling kecil sih, jadi wajar kalau capaiannya tinggi. Yang paling perlu diperhatiin itu Semarang (baru 69%, kurang sekitar Rp16,8 juta dari target) dan Magelang (70%, kurang Rp13,35 juta). Nilai per transaksi Semarang sebenarnya lumayan gede, rata-ratanya Rp410 ribu, tapi transaksinya baru 93, padahal butuh sekitar 134 biar nyampe target Rp55 juta. Magelang lebih berat lagi karena transaksinya paling sedikit (86). Jadi manajemen sebaiknya dorong jumlah penjualan di dua cabang ini, atau cek lagi apa target Semarang nggak kegedean.

In [7]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
